# Rainy-season onset periods on JASMIN with Dask Gateway

This notebook runs the rainy-season onset calculation over HEALPix rainfall grids using `xarray.apply_ufunc(..., dask="parallelized")` on a Dask Gateway cluster from the JASMIN Notebook Service.

It is based on the script you supplied, with these notebook-oriented changes:

- uses **Dask Gateway** rather than local Dask threads/processes;
- replaces `sys.argv` with editable notebook configuration;
- writes one NetCDF per `(simulation, zoom)`;
- keeps the full time axis in a single chunk for the gufunc core dimension, while spatial chunks remain parallel;
- computes and writes through the Dask distributed client;
- includes lightweight checks and dashboard links.

Before running: edit `SLURM_ACCOUNT`, paths, and optionally `WORKER_SETUP` if your notebook kernel uses a custom Python environment.

## 1. Imports

Run this from the JASMIN Notebook Service with a kernel/environment that has the same packages available on the LOTUS Dask workers. If you use a custom virtual environment, set `WORKER_SETUP` in the next section.

In [2]:
import datetime as dt
import os
import sys
import warnings
from pathlib import Path

import cartopy.crs as ccrs
import cmocean as cmo
import dask
import dask_gateway
import easygems.healpix as egh
import healpy as hp
import intake
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dask.distributed import progress

warnings.filterwarnings(
    "ignore",
    message=".*The return type of `Dataset.dims` will be changed.*",
    category=FutureWarning,
)

# Make local project modules importable. Adjust if this notebook is not one level
# below your project root.
project_root = Path.cwd().parent.resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils import hp_mods, hp_to_latlon
from onset_utils import onset_period_1d

print("imports done")
print(f"Project root: {project_root}")

ERROR 1: PROJ: proj_create_from_database: Open of /home/users/franmorr/.conda/envs/hk26_env/share/proj failed


imports done
Project root: /home/users/franmorr/hk26/hackathon-monsoons


## 2. User configuration

Set `SLURM_ACCOUNT` to the JASMIN/LOTUS Slurm account that should be charged for the Dask jobs. Leave `WORKER_SETUP = None` when the default `jaspy` environment is enough; otherwise point it at your environment activation command.

In [5]:
# Required: replace with your JASMIN Slurm account/project.
SLURM_ACCOUNT = "firstrains"  # e.g. "short4hr" is not an account; use your project account

# Optional: activate the same environment on Dask workers that your notebook kernel uses.
# Example: "source /home/users/franmorr/my_dask_env/bin/activate"
WORKER_SETUP = """
module load jaspy
source /home/users/franmorr/miniforge3/bin/activate
conda activate hk26_env
cd /home/users/franmorr/hk26/hackathon-monsoons/
export PYTHONUNBUFFERED=TRUE
export PYTHONPATH=$(pwd)
"""

# Cluster sizing. Start modestly and increase once the workflow is stable.
WORKER_CORES = 2
WORKER_MEMORY = 16.       # accepted values depend on current gateway options
ADAPT_MIN_WORKERS = 1
ADAPT_MAX_WORKERS = 8

# Calculation settings.
MAX_PERIODS = 2
ZOOMS = [3, 4, 5, 6, 7, 8]
# ZOOMS = [7]
SIMS = [
    # "ifs_tco3999-ng5_rcbmf_cf",
    "casesm2_10km_nocumulus",
    "icon_d3hp003",
    "nicam_gl11",
]

CATALOG_URL = "https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml"
CATALOG_BRANCH = "UK"

OUTDIR = Path("/home/users/franmorr/hk26/hackathon-monsoons/onset_identification/onset_dates/")
PLOT = False
IMAGE_DIR = Path("images")

# Useful while testing: set to a tuple of slices to reduce the domain after hp_to_latlon,
# for example TEST_DOMAIN = {"lat": slice(-20, 20), "lon": slice(0, 80)} if dims exist.
TEST_DOMAIN = None

OUTDIR.mkdir(parents=True, exist_ok=True)
if PLOT:
    IMAGE_DIR.mkdir(parents=True, exist_ok=True)

## 3. Start or reconnect to a JASMIN Dask Gateway cluster

This uses the JASMIN Notebook Service authentication path (`auth="jupyterhub"`). The scheduler and workers are submitted to LOTUS via Slurm.

[]

In [9]:
# def start_jasmin_dask_cluster(
#     slurm_account: str,
#     worker_cores: int = 2,
#     worker_memory: str | None = 16.,
#     worker_setup: str | None = None,
#     adapt_minimum: int = 1,
#     adapt_maximum: int = 8,
#     reuse_existing: bool = True,
# ):
#     if not slurm_account or slurm_account == "your-slurm-account-name":
#         raise ValueError("Set SLURM_ACCOUNT in the configuration cell before starting the cluster.")

#     gw = dask_gateway.Gateway("https://dask-gateway.jasmin.ac.uk", auth="jupyterhub")
#     options = gw.cluster_options()

#     # The exact option set can vary as the service evolves, so set only options
#     # which are available on the gateway.
#     if hasattr(options, "worker_cores"):
#         options.worker_cores = worker_cores
#     if worker_memory and hasattr(options, "worker_memory"):
#         options.worker_memory = worker_memory
#     if hasattr(options, "account"):
#         options.account = slurm_account
#     if worker_setup and hasattr(options, "worker_setup"):
#         options.worker_setup = worker_setup

#     clusters = gw.list_clusters()
#     if reuse_existing and clusters:
#         cluster = gw.connect(clusters[0].name)
#         print(f"Connected to existing cluster: {clusters[0].name}")
#     else:
#         cluster = gw.new_cluster(options, shutdown_on_close=False)
#         print("Requested a new Dask Gateway cluster")

#     cluster.adapt(minimum=adapt_minimum, maximum=adapt_maximum)
#     client = cluster.get_client()
#     print(f"Dashboard: {client.dashboard_link}")
#     return gw, cluster, client

# # Start the cluster. This cell may queue while LOTUS starts the scheduler/workers.
# gw, cluster, client = start_jasmin_dask_cluster(
#     slurm_account=SLURM_ACCOUNT,
#     worker_cores=WORKER_CORES,
#     worker_memory=WORKER_MEMORY,
#     worker_setup=WORKER_SETUP,
#     adapt_minimum=ADAPT_MIN_WORKERS,
#     adapt_maximum=ADAPT_MAX_WORKERS,
# )
# client

GatewayClusterError: Cluster '8a3979f081d146dc9dff68765a18e9b8' failed to start, see logs for more information

In [12]:
# Create a connection to dask-gateway.
gw = dask_gateway.Gateway("https://dask-gateway.jasmin.ac.uk", auth="jupyterhub")

# Inspect and change the options if required before creating your cluster.
options = gw.cluster_options()
options.worker_cores = 2
options.account = "firstrains"

# Create a Dask cluster, or, if one already exists, connect to it.
# This stage creates the scheduler job in Slurm, so it may take some
# time while your job queues.
clusters = gw.list_clusters()
if not clusters:
    cluster = gw.new_cluster(options, shutdown_on_close=False)
else:
    cluster = gw.connect(clusters[0].name)

# Create at least one worker, and allow your cluster to scale to three.
cluster.adapt(minimum=1, maximum=3)

# Get a Dask client.
client = cluster.get_client()


/home/users/franmorr/.conda/envs/hk26_env/lib/python3.14/site-packages/distributed/client.py:1607: VersionMismatchWarning: Mismatched versions found

+-------------+----------------+-----------------+---------+
| Package     | Client         | Scheduler       | Workers |
+-------------+----------------+-----------------+---------+
| cloudpickle | 3.1.2          | 3.1.1           | None    |
| dask        | 2026.3.0       | 2025.5.1        | None    |
| distributed | 2026.3.0       | 2025.5.1        | None    |
| lz4         | 4.4.5          | 4.4.4           | None    |
| msgpack     | 1.1.2          | 1.1.1           | None    |
| python      | 3.14.5.final.0 | 3.12.11.final.0 | None    |
| toolz       | 1.1.0          | 1.0.0           | None    |
| tornado     | 6.5.5          | 6.5.1           | None    |
+-------------+----------------+-----------------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [13]:
client

Connection method: Cluster object,Cluster type: dask_gateway.GatewayCluster
Dashboard: /_jasmin/proxy/dask-gateway/clusters/84832add755f40d3b192142eedcf88b3/status,


## 4. Open the Digital Earths catalog

In [14]:
cat = intake.open_catalog(CATALOG_URL)[CATALOG_BRANCH]
print(cat)

<Intake catalog: UK>


## 5. Helper functions

`apply_onset_periods` is the key part of the calculation. The gufunc needs the full `time` core dimension in a single chunk, so the notebook rechunks daily precipitation with `time=-1` immediately before `xr.apply_ufunc`. Spatial chunks remain distributed across workers.

In [15]:
def open_simulation_dataset(sim_cat, sim: str, zoom: int) -> xr.Dataset:
    """Open one simulation/zoom from the hackathon intake catalog."""
    if "hk26" in sim:
        ds = sim_cat(zoom=zoom, time="PT1H").to_dask().pipe(hp_mods)
    else:
        if "icon" in sim:
            ds = sim_cat(zoom=zoom, time_method="inst", time="PT1H").to_dask().pipe(egh.attach_coords)
        else:
            ds = sim_cat(zoom=zoom, time="PT1H").to_dask().pipe(egh.attach_coords)
    return ds


def daily_precip_latlon(ds: xr.Dataset, zoom: int) -> xr.DataArray:
    """Convert HEALPix dataset to lat/lon and return daily precipitation."""
    ds_latlon = hp_to_latlon(ds, zoom)

    if TEST_DOMAIN:
        existing_indexers = {k: v for k, v in TEST_DOMAIN.items() if k in ds_latlon.dims or k in ds_latlon.coords}
        if existing_indexers:
            ds_latlon = ds_latlon.sel(**existing_indexers)

    pp_latlon = ds_latlon.pr.resample(time="1D").mean()

    # Convert from per-second flux-style rates to mm h-1, following your script.
    pp_latlon = pp_latlon * 3600
    pp_latlon.attrs["units"] = "mm h-1"

    # xarray gufunc core dims must be a single chunk unless allow_rechunk=True.
    # Keeping time in one chunk is usually safer here because daily time length is modest.
    pp_latlon = pp_latlon.chunk({"time": -1})
    return pp_latlon


def apply_onset_periods(pp_latlon: xr.DataArray, max_periods: int = MAX_PERIODS) -> xr.Dataset:
    """Apply onset_period_1d over every non-time pixel."""
    first_days, last_days = xr.apply_ufunc(
        onset_period_1d,
        pp_latlon,
        pp_latlon["time"],
        input_core_dims=[["time"], ["time"]],
        output_core_dims=[["period"], ["period"]],
        kwargs={
            "max_periods": max_periods,
            "max_dry_frac_rainfall": 0.1,
            "refine": True,
            # "precip_threshold": 0.05,
            # "intensity_threshold": "60%",
        },
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float, float],
        dask_gufunc_kwargs={
            "output_sizes": {"period": max_periods},
        },
    )

    period_coord = np.arange(max_periods)
    first_days = first_days.assign_coords(period=period_coord).rename("first_day_of_period")
    last_days = last_days.assign_coords(period=period_coord).rename("last_day_of_period")

    dwtps = xr.merge([first_days, last_days])

    # Remove attrs which can trip NetCDF writing.
    dwtps.attrs.pop("hiopy::enable", None)
    for var in dwtps.variables:
        dwtps[var].attrs.pop("hiopy::enable", None)

    return dwtps


def encoding_for_netcdf(ds: xr.Dataset) -> dict:
    """Compression/chunking hints for NetCDF output."""
    encoding = {}
    for name, da in ds.data_vars.items():
        encoding[name] = {
            "zlib": True,
            "complevel": 4,
            "dtype": "float32",
            "_FillValue": np.float32(np.nan),
        }
    return encoding


def run_one(sim: str, zoom: int, overwrite: bool = False) -> xr.Dataset:
    """Compute one simulation/zoom and write it to NetCDF."""
    outfile = OUTDIR / f"{sim}_zoom_{zoom}_dwtps.nc"
    if outfile.exists() and not overwrite:
        print(f"{outfile} exists, opening existing file")
        return xr.open_dataset(outfile)

    print(f"Opening {sim=} {zoom=}")
    sim_cat = cat[sim]
    ds = open_simulation_dataset(sim_cat, sim, zoom)
    pp_latlon = daily_precip_latlon(ds, zoom)

    print("Daily precip:", pp_latlon)
    print("Chunks:", pp_latlon.chunks)

    dwtps = apply_onset_periods(pp_latlon, MAX_PERIODS)
    print("Output dataset:", dwtps)

    # Build a delayed write and let the distributed client execute it.
    delayed_write = dwtps.to_netcdf(
        outfile,
        mode="w",
        compute=False,
        encoding=encoding_for_netcdf(dwtps),
    )
    future = client.compute(delayed_write)
    progress(future)
    future.result()
    print(f"Wrote {outfile}")

    return xr.open_dataset(outfile)

## 6. Optional smoke test

Run one low-zoom calculation first. For a very quick test, set `TEST_DOMAIN` in the configuration cell and rerun the helper cells.

In [16]:
# Example smoke test. Uncomment to run.
test_result = run_one(SIMS[0], ZOOMS[3], overwrite=False)
test_result

Opening sim='casesm2_10km_nocumulus' zoom=6


## 7. Run all simulations and zooms

This loops over all configured simulations and zooms. Existing outputs are opened and skipped unless `OVERWRITE = True`.

In [ ]:
OVERWRITE = False

labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)", "(h)", "(i)", "(j)", "(k)"]
projection = ccrs.PlateCarree()

for sim in SIMS:
    print("=" * 80)
    print(sim)

    if PLOT:
        fig, axes = plt.subplots(
            len(ZOOMS),
            2,
            subplot_kw={"projection": projection},
            layout="constrained",
        )

    label_ix = 0
    for zoom_ix, zoom in enumerate(ZOOMS):
        print("-" * 80)
        print(sim, zoom)
        dwtps = run_one(sim, zoom, overwrite=OVERWRITE)

        if PLOT:
            for ix in range(MAX_PERIODS):
                label = labels[label_ix]
                ax = axes[zoom_ix, ix]
                ax.set_global()
                ax.coastlines()
                m = (dwtps.first_day_of_period.sel(period=ix) % 365).plot(
                    ax=ax,
                    vmin=0,
                    vmax=365,
                    cmap=cmo.cm.phase,
                    add_colorbar=False,
                )
                ax.set_title(f"{label} zoom={zoom}")
                label_ix += 1

    if PLOT:
        fig.suptitle(sim)
        fig.colorbar(
            m,
            ax=axes.ravel().tolist(),
            orientation="horizontal",
            fraction=0.05,
            pad=0.07,
            shrink=0.7,
            label="day of year",
        )
        fig.savefig(IMAGE_DIR / f"{sim}_zooms_{''.join([str(zoom) for zoom in ZOOMS])}.png", dpi=150)
        plt.close(fig)

## 8. Inspect outputs

In [ ]:
outputs = sorted(OUTDIR.glob("*_dwtps.nc"))
print(f"Found {len(outputs)} onset output files in {OUTDIR}")
outputs[:10]

In [ ]:
# Open a combined view of outputs if useful. This assumes consistent dimensions/coords.
# You may prefer to inspect individual files if zooms have different grids.
# sample = xr.open_dataset(outputs[0])
# sample

## 9. Shut down the cluster when finished

Run this when all computations have completed and you no longer need the Dask workers.

In [ ]:
# cluster.shutdown()

## Notes and tuning tips

- `apply_ufunc` with `input_core_dims=[["time"], ["time"]]` requires the whole daily time series for each pixel. This notebook keeps `time` in a single chunk after daily resampling.
- If memory pressure is high, reduce spatial chunk sizes before `apply_onset_periods`. For example, add `pp_latlon = pp_latlon.chunk({"lat": 64, "lon": 64, "time": -1})` if those dimensions exist.
- If workers fail because custom project modules are missing (`utils`, `onset_utils`), ensure the notebook path/project root is visible on the workers, or package those utilities into the environment used by `WORKER_SETUP`.
- For larger runs, increase `ADAPT_MAX_WORKERS` gradually and watch the Dask dashboard.